# Phase 5 - XAI vs Pseudo-Concept Evaluation - Pilot 100

This notebook evaluates whether XAI saliency maps align with the pseudo-concept maps generated from the lesion masks and images.

It expects the pilot outputs from the previous notebooks:

```text
data/pilot/pilot_subset_100.csv

outputs/pilot/manifests/pilot_xai_manifest.csv
outputs/pilot/manifests/pilot_pseudo_concepts_manifest.csv

outputs/pilot/xai_maps/
outputs/pilot/pseudo_concepts/npz/
```

Expected metric rows:

```text
100 images × 3 XAI methods × 3 pseudo-concepts = 900 rows
```

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

## 1. Configuration

In [ ]:
# If your notebook is in src/, ROOT = Path("..").resolve() is usually correct.
# If you run this notebook from the repository root, change ROOT = Path(".").resolve().

ROOT = Path("..").resolve()

DATA_DIR = ROOT / "data"
OUTPUTS_DIR = ROOT / "outputs"

PILOT_CSV = DATA_DIR / "pilot" / "pilot_subset_100.csv"

XAI_MANIFEST = OUTPUTS_DIR / "pilot" / "manifests" / "pilot_xai_manifest.csv"
PSEUDO_MANIFEST = OUTPUTS_DIR / "pilot" / "manifests" / "pilot_pseudo_concepts_manifest.csv"

PHASE5_DIR = OUTPUTS_DIR / "phase5_pilot"
METRICS_DIR = PHASE5_DIR / "metrics"
FIG_DIR = PHASE5_DIR / "figures"

METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

XAI_METHODS = ["gradcam", "lime", "shap"]

# These keys must match the keys saved inside the pseudo-concept .npz files.
CONCEPT_KEYS = {
    "asymmetry": "asymmetry",
    "border_default": "border_irregularity",
    "border_w8_dil2_sigma5": "border_w8_dil2_sigma5",
    "border_w12_dil4_sigma8": "border_w12_dil4_sigma8",
    "border_w16_dil6_sigma10": "border_w16_dil6_sigma10",
    "border_w16_dil6_sigma10_dist": "border_w16_dil6_sigma10_dist",

    "colour_heterogeneity": "colour_heterogeneity",
}

TOP_K_PERCENT = 20
EPS = 1e-8

print("ROOT:", ROOT)
print("Pilot CSV:", PILOT_CSV, "exists:", PILOT_CSV.exists())
print("XAI manifest:", XAI_MANIFEST, "exists:", XAI_MANIFEST.exists())
print("Pseudo manifest:", PSEUDO_MANIFEST, "exists:", PSEUDO_MANIFEST.exists())

## 2. Load and merge pilot manifests

In [ ]:
def normalise_dataset_name(x):
    x = str(x).strip().lower()
    aliases = {
        "ham": "ham10000",
        "ham10000": "ham10000",
        "isic": "isic2018",
        "isic2018": "isic2018",
    }
    return aliases.get(x, x)


def ensure_exists(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")


ensure_exists(PILOT_CSV, "Pilot CSV")
ensure_exists(XAI_MANIFEST, "XAI manifest")
ensure_exists(PSEUDO_MANIFEST, "Pseudo-concept manifest")

pilot = pd.read_csv(PILOT_CSV)
xai_manifest = pd.read_csv(XAI_MANIFEST)
pseudo_manifest = pd.read_csv(PSEUDO_MANIFEST)

for name, df in [
    ("pilot", pilot),
    ("xai_manifest", xai_manifest),
    ("pseudo_manifest", pseudo_manifest),
]:
    if "dataset" not in df.columns or "stem" not in df.columns:
        raise ValueError(f"{name} must contain columns: dataset, stem")

    df["dataset"] = df["dataset"].map(normalise_dataset_name)
    df["stem"] = df["stem"].astype(str)

print("Pilot rows:", len(pilot))
print("XAI manifest rows:", len(xai_manifest))
print("Pseudo manifest rows:", len(pseudo_manifest))

display(pilot.head())
display(xai_manifest.head())
display(pseudo_manifest.head())

In [ ]:
pilot_resolved = (
    pilot
    .merge(
        xai_manifest,
        on=["dataset", "stem"],
        how="left",
        suffixes=("", "_xai"),
    )
    .merge(
        pseudo_manifest,
        on=["dataset", "stem"],
        how="left",
        suffixes=("", "_pseudo"),
    )
)

print("Merged pilot rows:", len(pilot_resolved))
display(pilot_resolved.head())

for method in XAI_METHODS:
    col = f"xai_{method}_path"
    if col in pilot_resolved.columns:
        print(f"Rows with {col}:", pilot_resolved[col].notna().sum())
    else:
        print(f"Missing column: {col}")

if "pseudo_npz_path" in pilot_resolved.columns:
    print("Rows with pseudo_npz_path:", pilot_resolved["pseudo_npz_path"].notna().sum())
else:
    print("Missing column: pseudo_npz_path")

## 3. Validate file paths

In [ ]:
def resolve_path(p):
    if p is None or pd.isna(p):
        return None

    p = Path(str(p))

    if p.is_absolute():
        return p

    # First try relative to current notebook working directory.
    if p.exists():
        return p.resolve()

    # Then try relative to repository root.
    p2 = ROOT / p
    if p2.exists():
        return p2.resolve()

    return p


path_check_records = []

for _, row in pilot_resolved.iterrows():
    rec = {
        "dataset": row["dataset"],
        "stem": row["stem"],
    }

    for method in XAI_METHODS:
        col = f"xai_{method}_path"
        p = resolve_path(row.get(col)) if col in pilot_resolved.columns else None
        rec[f"{method}_exists"] = bool(p is not None and Path(p).exists())

    p = resolve_path(row.get("pseudo_npz_path")) if "pseudo_npz_path" in pilot_resolved.columns else None
    rec["pseudo_exists"] = bool(p is not None and Path(p).exists())

    path_check_records.append(rec)

path_check = pd.DataFrame(path_check_records)

display(path_check.head())

print("File availability summary:")
display(path_check.drop(columns=["dataset", "stem"]).sum().to_frame("count"))

missing_any = path_check[
    ~path_check[[f"{m}_exists" for m in XAI_METHODS] + ["pseudo_exists"]].all(axis=1)
]

print("Rows missing at least one required file:", len(missing_any))
display(missing_any.head(20))

In [ ]:
# Optional quick check: inspect the first pseudo-concept NPZ file.
first_pseudo_raw = pilot_resolved["pseudo_npz_path"].dropna().iloc[0]
first_pseudo = resolve_path(first_pseudo_raw)

with np.load(first_pseudo, allow_pickle=False) as data:
    print("First pseudo-concept file:", first_pseudo)
    print("Available keys:", data.files)


## 4. Map loading and metric helpers

In [ ]:
def normalize_map(x):
    arr = np.asarray(x)
    arr = np.squeeze(arr)

    # Convert common channel-first or channel-last attribution arrays to 2D.
    if arr.ndim == 3:
        if arr.shape[0] in [1, 3, 4]:
            arr = np.mean(np.abs(arr), axis=0)
        elif arr.shape[-1] in [1, 3, 4]:
            arr = np.mean(np.abs(arr), axis=-1)

    if arr.ndim != 2:
        raise ValueError(f"Expected 2D map after conversion, got shape {arr.shape}")

    arr = arr.astype(np.float32)

    if not np.isfinite(arr).all():
        arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

    arr = arr - arr.min()
    denom = arr.max() + EPS
    arr = arr / denom

    return arr


def load_map_file(path, key=None):
    path = resolve_path(path)
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(path)

    suffix = path.suffix.lower()

    if suffix == ".npy":
        return normalize_map(np.load(path, allow_pickle=False))

    if suffix == ".npz":
        data = np.load(path, allow_pickle=False)

        if key is not None:
            if key not in data.files:
                raise KeyError(
                    f"Key '{key}' not found in {path}. Available keys: {data.files}"
                )
            return normalize_map(data[key])

        return normalize_map(data[data.files[0]])

    img = Image.open(path).convert("L")
    return normalize_map(np.asarray(img))


def topk_binary(x, top_k_percent=20):
    x = normalize_map(x)
    threshold = np.percentile(x, 100 - top_k_percent)
    return x >= threshold


def binary_from_map(x, threshold=0.5):
    x = normalize_map(x)
    return x >= threshold


def iou_score(a, b):
    a = np.asarray(a).astype(bool)
    b = np.asarray(b).astype(bool)
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter / (union + EPS))


def dice_score(a, b):
    a = np.asarray(a).astype(bool)
    b = np.asarray(b).astype(bool)
    inter = np.logical_and(a, b).sum()
    return float((2 * inter) / (a.sum() + b.sum() + EPS))


def saliency_inside_ratio(saliency, region):
    saliency = normalize_map(saliency)
    region = np.asarray(region).astype(bool)

    total = saliency.sum()
    if total <= EPS:
        return 0.0

    return float(saliency[region].sum() / (total + EPS))


def mean_inside_outside_ratio(saliency, region):
    saliency = normalize_map(saliency)
    region = np.asarray(region).astype(bool)

    if region.sum() == 0:
        return 0.0

    inside = saliency[region].mean()

    if (~region).sum() == 0:
        return np.nan

    outside = saliency[~region].mean()

    return float(inside / (outside + EPS))


def pearson_corr(a, b):
    a = normalize_map(a).ravel()
    b = normalize_map(b).ravel()

    if a.std() <= EPS or b.std() <= EPS:
        return 0.0

    return float(np.corrcoef(a, b)[0, 1])


## 5. Inspect pseudo-concept NPZ keys

In [ ]:
if "pseudo_npz_path" not in pilot_resolved.columns:
    raise ValueError("pseudo_npz_path column is missing from the merged dataframe.")

first_pseudo = pilot_resolved["pseudo_npz_path"].dropna().iloc[0]
first_pseudo = resolve_path(first_pseudo)

print("First pseudo-concept file:", first_pseudo)

with np.load(first_pseudo, allow_pickle=False) as data:
    print("Available keys:", data.files)

print("Expected keys:")
for concept_name, key in CONCEPT_KEYS.items():
    print(f"{concept_name}: {key}")

## 6. Compute XAI vs lesion mask metrics


In [ ]:
# ============================================================
# XAI vs lesion mask evaluation

mask_records = []

for _, row in tqdm(pilot_resolved.iterrows(), total=len(pilot_resolved)):

    pseudo_path = row.get("pseudo_npz_path")

    if pseudo_path is None or pd.isna(pseudo_path):
        continue

    try:
        pseudo_path = resolve_path(pseudo_path)

        with np.load(pseudo_path, allow_pickle=False) as data:
            mask = data["mask"].astype(np.float32)

    except Exception as e:
        print("Could not load mask from pseudo NPZ:", pseudo_path, e)
        continue

    mask_bin = binary_from_map(mask, threshold=0.5)

    for method in XAI_METHODS:
        xai_path = row.get(f"xai_{method}_path")

        if xai_path is None or pd.isna(xai_path):
            continue

        try:
            xai_map = load_map_file(resolve_path(xai_path))
        except Exception as e:
            print("Could not load XAI:", xai_path, e)
            continue

        xai_bin = topk_binary(xai_map, TOP_K_PERCENT)

        mask_records.append({
            "dataset": row["dataset"],
            "stem": row["stem"],
            "xai_method": method,
            "iou": iou_score(xai_bin, mask_bin),
            "dice": dice_score(xai_bin, mask_bin),
            "sir": saliency_inside_ratio(xai_map, mask_bin),
            "inside_outside_ratio": mean_inside_outside_ratio(xai_map, mask_bin),
            "mask_area_ratio": mask_bin.mean(),
            "xai_area_ratio": xai_bin.mean(),
        })

mask_metrics = pd.DataFrame(mask_records)

print("Mask evaluation rows:", len(mask_metrics))
display(mask_metrics.head())

In [ ]:
# Optional diagnostics after mask evaluation.
print("pilot_resolved shape:", pilot_resolved.shape)
print("XAI path columns:", [c for c in pilot_resolved.columns if c.startswith("xai_") and c.endswith("_path")])
print("Pseudo rows:", pilot_resolved["pseudo_npz_path"].notna().sum())
print("Mask metric rows:", len(mask_metrics))


## 7. Compute XAI vs pseudo-concept metrics

In [ ]:
records = []
load_errors = []

for _, row in tqdm(pilot_resolved.iterrows(), total=len(pilot_resolved)):
    dataset = row["dataset"]
    stem = str(row["stem"])

    pseudo_path = row.get("pseudo_npz_path")
    if pseudo_path is None or pd.isna(pseudo_path):
        continue

    pseudo_path = resolve_path(pseudo_path)

    for method in XAI_METHODS:
        xai_col = f"xai_{method}_path"

        if xai_col not in pilot_resolved.columns:
            continue

        xai_path = row.get(xai_col)

        if xai_path is None or pd.isna(xai_path):
            continue

        xai_path = resolve_path(xai_path)

        try:
            xai_map = load_map_file(xai_path)
            xai_bin = topk_binary(xai_map, TOP_K_PERCENT)
        except Exception as e:
            load_errors.append({
                "dataset": dataset,
                "stem": stem,
                "type": "xai",
                "method": method,
                "path": str(xai_path),
                "error": str(e),
            })
            continue

        for concept_name, concept_key in CONCEPT_KEYS.items():
            try:
                concept_map = load_map_file(pseudo_path, key=concept_key)
                concept_bin = binary_from_map(concept_map, threshold=0.5)
            except Exception as e:
                load_errors.append({
                    "dataset": dataset,
                    "stem": stem,
                    "type": "concept",
                    "concept": concept_name,
                    "key": concept_key,
                    "path": str(pseudo_path),
                    "error": str(e),
                })
                continue

            records.append({
                "dataset": dataset,
                "stem": stem,
                "xai_method": method,
                "concept": concept_name,
                "top_k_percent": TOP_K_PERCENT,
                "iou": iou_score(xai_bin, concept_bin),
                "dice": dice_score(xai_bin, concept_bin),
                "sir": saliency_inside_ratio(xai_map, concept_bin),
                "inside_outside_ratio": mean_inside_outside_ratio(xai_map, concept_bin),
                "pearson_corr": pearson_corr(xai_map, concept_map),
                "concept_area_ratio": float(concept_bin.mean()),
                "xai_area_ratio": float(xai_bin.mean()),
                "xai_path": str(xai_path),
                "pseudo_npz_path": str(pseudo_path),
            })

metrics = pd.DataFrame(records)
errors = pd.DataFrame(load_errors)

metrics_path = METRICS_DIR / "phase5_pilot_100_xai_concept_metrics.csv"
errors_path = METRICS_DIR / "phase5_pilot_100_load_errors.csv"

metrics.to_csv(metrics_path, index=False)
errors.to_csv(errors_path, index=False)

print("Metric rows:", len(metrics))
print("Expected if complete:", len(pilot_resolved) * len(XAI_METHODS) * len(CONCEPT_KEYS))
print("Saved metrics:", metrics_path)

print("Load errors:", len(errors))
print("Saved errors:", errors_path)

display(metrics.head())
display(errors.head())

## 8. Summary tables

In [ ]:

# ============================================================
# XAI vs lesion mask summary

if len(mask_metrics) == 0:
    raise ValueError("No mask metrics were generated.")

mask_summary = (
    mask_metrics
    .groupby(["xai_method"])
    .agg(
        n=("stem", "count"),
        mean_iou=("iou", "mean"),
        mean_dice=("dice", "mean"),
        mean_sir=("sir", "mean"),
        mean_inside_outside_ratio=("inside_outside_ratio", "mean"),
        mean_mask_area=("mask_area_ratio", "mean"),
        mean_xai_area=("xai_area_ratio", "mean"),
    )
    .reset_index()
    .sort_values("mean_sir", ascending=False)
)

mask_summary_path = (
    METRICS_DIR
    / "phase5_pilot_100_mask_summary_by_method.csv"
)

mask_summary.to_csv(mask_summary_path, index=False)

print("Saved mask summary:", mask_summary_path)

display(mask_summary)

In [ ]:
# ============================================================
# XAI vs pseudo-concept summary
# ============================================================

if len(metrics) == 0:
    raise ValueError("No pseudo-concept metrics were generated.")

summary = (
    metrics
    .groupby(["xai_method", "concept"])
    .agg(
        n=("stem", "count"),
        mean_iou=("iou", "mean"),
        mean_dice=("dice", "mean"),
        mean_sir=("sir", "mean"),
        mean_inside_outside_ratio=("inside_outside_ratio", "mean"),
        mean_pearson_corr=("pearson_corr", "mean"),
        mean_concept_area=("concept_area_ratio", "mean"),
    )
    .reset_index()
    .sort_values(["concept", "mean_sir"], ascending=[True, False])
)

summary_path = METRICS_DIR / "phase5_pilot_100_summary_by_method_concept.csv"
summary.to_csv(summary_path, index=False)

print("Saved pseudo-concept summary:", summary_path)
display(summary)



dataset_summary = (
    metrics
    .groupby(["dataset", "xai_method", "concept"])
    .agg(
        n=("stem", "count"),
        mean_iou=("iou", "mean"),
        mean_dice=("dice", "mean"),
        mean_sir=("sir", "mean"),
        mean_pearson_corr=("pearson_corr", "mean"),
    )
    .reset_index()
    .sort_values(["dataset", "concept", "mean_sir"], ascending=[True, True, False])
)

dataset_summary_path = METRICS_DIR / "phase5_pilot_100_summary_by_dataset.csv"
dataset_summary.to_csv(dataset_summary_path, index=False)

print("Saved dataset summary:", dataset_summary_path)
display(dataset_summary)

In [ ]:
# ============================================================
# Pseudo-concept performance by lesion size class

size_summary = (
    metrics
    .merge(
        pilot_resolved[["dataset", "stem", "mask_size_class"]],
        on=["dataset", "stem"],
        how="left"
    )
    .groupby(["mask_size_class", "xai_method", "concept"])
    .agg(
        n=("stem", "count"),
        mean_iou=("iou", "mean"),
        mean_dice=("dice", "mean"),
        mean_sir=("sir", "mean"),
        mean_pearson_corr=("pearson_corr", "mean"),
        mean_concept_area=("concept_area_ratio", "mean"),
    )
    .reset_index()
    .sort_values(["concept", "mask_size_class", "mean_sir"], ascending=[True, True, False])
)

size_summary_path = METRICS_DIR / "phase5_pilot_100_summary_by_size_class.csv"
size_summary.to_csv(size_summary_path, index=False)

display(size_summary)
print("Saved:", size_summary_path)




normal_summary = (
    size_summary[size_summary["mask_size_class"] == "normal"]
    .sort_values(["concept", "mean_sir"], ascending=[True, False])
)

display(normal_summary)


for concept in ["asymmetry", "border_w16_dil6_sigma10"]:
    subset = size_summary[size_summary["concept"] == concept]

    pivot = subset.pivot_table(
        index="mask_size_class",
        columns="xai_method",
        values="mean_sir",
        aggfunc="mean"
    )

    ax = pivot.plot(kind="bar", figsize=(8, 4))
    ax.set_title(f"Pseudo-concept SIR by lesion size - {concept}")
    ax.set_ylabel("mean_sir")
    ax.set_xlabel("mask_size_class")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

## 9. Best and worst examples

In [ ]:
for metric_name in ["dice", "sir", "pearson_corr"]:
    print("\n", "=" * 80)
    print("Metric:", metric_name)
    print("Top 10")
    display(metrics.sort_values(metric_name, ascending=False).head(10))
    print("Bottom 10")
    display(metrics.sort_values(metric_name, ascending=True).head(10))

## 10. Bar plots

This older bar-plot cell was replaced by the two separate plotting cells below: pseudo-concepts and lesion masks.


In [ ]:
# ============================================================
# Bar plots: XAI vs pseudo-concepts only
# ============================================================

plot_metrics = [
    "mean_dice",
    "mean_sir",
]

if "mean_pearson_corr" in summary.columns:
    plot_metrics.append("mean_pearson_corr")

for metric_name in plot_metrics:
    if metric_name not in summary.columns:
        print(f"Skipping {metric_name}: not found in summary")
        continue

    pivot = summary.pivot_table(
        index="concept",
        columns="xai_method",
        values=metric_name,
        aggfunc="mean"
    )

    ax = pivot.plot(kind="bar", figsize=(10, 5))
    ax.set_title(f"Pilot 100 - pseudo-concepts - {metric_name}")
    ax.set_ylabel(metric_name)
    ax.set_xlabel("Pseudo-concept")
    ax.legend(title="XAI method")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()

    fig_path = FIG_DIR / f"phase5_pilot_100_pseudo_{metric_name}.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()

    print("Saved:", fig_path)

In [ ]:
# ============================================================
# Bar plots: XAI vs lesion mask only
# ============================================================

mask_plot_metrics = [
    "mean_iou",
    "mean_dice",
    "mean_sir",
]

for metric_name in mask_plot_metrics:
    if metric_name not in mask_summary.columns:
        print(f"Skipping {metric_name}: not found in mask_summary")
        continue

    ax = mask_summary.plot(
        x="xai_method",
        y=metric_name,
        kind="bar",
        legend=False,
        figsize=(7, 4)
    )

    ax.set_title(f"Pilot 100 - lesion mask - {metric_name}")
    ax.set_ylabel(metric_name)
    ax.set_xlabel("XAI method")
    plt.xticks(rotation=0)
    plt.tight_layout()

    fig_path = FIG_DIR / f"phase5_pilot_100_mask_{metric_name}.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()

    print("Saved:", fig_path)

## 11. Visual sanity-check grids

In [ ]:
def find_image_path_from_row(row):
    # Try common path columns from the original pilot/preprocessing manifests.
    candidates = [
        "image_path",
        "img_path",
        "resized_image_path",
        "preprocessed_image_path",
        "path",
    ]

    for col in candidates:
        if col in row.index and pd.notna(row[col]):
            p = resolve_path(row[col])
            if p is not None and Path(p).exists():
                return Path(p)

    return None


def load_rgb_image(path):
    return np.asarray(Image.open(path).convert("RGB"))


def show_or_blank(ax, arr, title="", cmap=None, alpha=1.0):
    ax.set_title(title, fontsize=8)
    ax.axis("off")

    if arr is None:
        ax.text(0.5, 0.5, "missing", ha="center", va="center")
        return

    ax.imshow(arr, cmap=cmap, alpha=alpha, vmin=0 if cmap else None, vmax=1 if cmap else None)


def make_visual_grid(rows, method="gradcam", concept_keys=CONCEPT_KEYS, max_images=8, save_path=None):
    rows = rows.head(max_images)
    n = len(rows)

    if n == 0:
        print("No rows to plot.")
        return

    ncols = 2 + len(concept_keys)

    fig, axes = plt.subplots(n, ncols, figsize=(3.2 * ncols, 3.2 * n))

    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_i, (_, row) in enumerate(rows.iterrows()):
        stem = str(row["stem"])
        dataset = str(row["dataset"])

        image_path = find_image_path_from_row(row)
        image = load_rgb_image(image_path) if image_path is not None else None

        xai = None
        xai_col = f"xai_{method}_path"
        if xai_col in row.index and pd.notna(row[xai_col]):
            try:
                xai = load_map_file(row[xai_col])
            except Exception as e:
                print(f"Could not load XAI for {dataset}/{stem}: {e}")

        pseudo_path = row.get("pseudo_npz_path")

        show_or_blank(axes[row_i, 0], image, f"{dataset}\n{stem}")

        if image is not None:
            axes[row_i, 1].imshow(image)
        show_or_blank(axes[row_i, 1], xai, method, cmap="hot", alpha=0.45)

        for concept_i, (concept_name, concept_key) in enumerate(concept_keys.items(), start=2):
            concept_map = None

            if pseudo_path is not None and pd.notna(pseudo_path):
                try:
                    concept_map = load_map_file(pseudo_path, key=concept_key)
                except Exception as e:
                    print(f"Could not load concept {concept_key} for {dataset}/{stem}: {e}")

            show_or_blank(
                axes[row_i, concept_i],
                concept_map,
                concept_name,
                cmap="viridis",
            )

    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print("Saved:", save_path)

    plt.show()

In [ ]:
# Visualise a few examples for each XAI method.

example_rows = pilot_resolved.copy()

for method in XAI_METHODS:
    print("\n", "=" * 80)
    print("Visual grid:", method)

    save_path = FIG_DIR / f"phase5_pilot_100_visual_grid_{method}.png"

    make_visual_grid(
        example_rows,
        method=method,
        max_images=8,
        save_path=save_path,
    )

## 12. Export merged manifest for traceability

In [ ]:
resolved_path = METRICS_DIR / "phase5_pilot_100_resolved_manifest.csv"
pilot_resolved.to_csv(resolved_path, index=False)

print("Saved resolved manifest:", resolved_path)
print("Done.")